In [0]:
SELECT *
FROM read_files(r"/Volumes/projects/muranga/projects")

Project Name,Description,Sub-County(s),Ward(s),Sector(s),Status,Approved Cost,Disbursed,Expenditure,Completion %,Fiscal Year,Start Date,End Date,Development Partners,Partner Components,_rescued_data
French beans and marketing project,Provide production inputs and improve market access for french beans,Gatanga; Kandara; Kangema; Kigumo; Kiharu; Maragua; Mathioya,Gatanga; Ithanga; Kakuzi/Mitubiri; Kariara; Kihumbu-ini; Mugumo-ini; Gaichanjiru; Ithiru; Kagundu-ini; Muruka; Ng'araria; Ruchu; Kanyenyaini; Muguru; Rwathia; Kahumbu; Kangari; Kigumo; Kinyona; Muthithi; Gaturi; Mbiri; Mugoiri; Murarandia; Township; Wangu; Ichagaki; Kamahuha; Kambiti; Kimorori/Wempa; Makuyu; Nginda; Gitugi; Kamacharia; Kiru,Agriculture,Planned,6000000.0,6000000.0,6000000.0,1.0,FY 13-14,2014-01-01,2015-01-01,null,null,null
Kenyatta Agriculture Farmers Training Centre (Mariira farm ). Staff and farmers capacity building,Capacity building of staff and farmers on agricultural technologies,Gatanga; Kandara; Kangema; Kigumo; Kiharu; Maragua; Mathioya,Gatanga; Ithanga; Kakuzi/Mitubiri; Kariara; Kihumbu-ini; Mugumo-ini; Gaichanjiru; Ithiru; Kagundu-ini; Muruka; Ng'araria; Ruchu; Kanyenyaini; Muguru; Rwathia; Kahumbu; Kangari; Kigumo; Kinyona; Muthithi; Gaturi; Mbiri; Mugoiri; Murarandia; Township; Wangu; Ichagaki; Kamahuha; Kambiti; Kimorori/Wempa; Makuyu; Nginda; Gitugi; Kamacharia; Kiru,Agriculture,Ongoing,1120000.0,1120000.0,1120000.0,1.0,FY 13-14,2014-07-01,2014-07-01,null,null,null
Gikindu (Thailand) Farmers Dry land Training Centre project,Project to develop support infrastructure for stakeholders training,Kiharu,Mbiri,Agriculture,Stalled,2.05E8,0.0,1.0E8,0.5,FY 13-14,2014-06-01,2014-06-01,null,null,null
Semi-permanent French beans collection centres in Kigumo and Kandara,Establish infrastructure for aggregation of fresh farm produce in readiness for market offtake,Kandara; Kigumo,Ithiru; Kigumo,Agriculture,Completed,4000000.0,4000000.0,4000000.0,100.0,FY 14-15,2014-01-01,2015-01-01,null,null,null
Gikindu (Thailand) Farmers Dry land Training Centre project,Project to develop support infrastructure for stakeholders training,Kiharu,Mbiri,Agriculture,Stalled,5000000.0,5000000.0,5000000.0,0.5,FY 14-15,2014-06-01,2015-06-01,null,null,null
Ajibika Irrigation project,An initiative designed to convey water to small holder farms to boost food and nutrition security.,Kandara,Ithiru; Ruchu,Agriculture,Completed,1.0E8,1.0E8,1.0E8,100.0,FY 14-15,2014-06-01,2015-01-01,null,null,null
Kanyenyaini Irrigation project,null,Kangema,Kanyenyaini,Agriculture,Stalled,8.7E7,8.7E7,8.7E7,0.58,FY 14-15,2014-06-01,2015-01-01,null,null,null
Kimathi Githuri irrigation project-Food security programme (rice and Vegetables),An initiative designed to convey water to small holder farms to boost food and nutrition security.,Kiharu,Gaturi; Mbiri,Agriculture,Completed,1.0E7,1.0E7,1.0E7,100.0,FY 14-15,1905-07-07,2017-01-01,null,null,null
Food a nd nutrition security -Ria Komu- Thika Ridge irrigation Project,An initiative designed to convey water to small holder farms to boost food and nutrition security.,Gatanga,Kihumbu-ini,Agriculture,Completed,8000000.0,5000000.0,5000000.0,100.0,FY 14-15,null,2019-01-01,IFAD,IFAD: 5000000.00,null
Agriculture Sector Development Support Programme Phase one,"Project contributed to transformation of Dairy ,French Beans and Banana Value chains to commercially oriented enterprises that ensure sustainable food and nutrition security.",Gatanga; Kandara; Kangema; Kigumo; Kiharu; Maragua; Mathioya,Gatanga; Ithanga; Kakuzi/Mitubiri; Kariara; Kihumbu-ini; Mugumo-ini; Gaichanjiru; Ithiru; Kagundu-ini; Muruka; Ng'araria; Ruchu; Kanyenyaini; Muguru; Rwathia; Kahumbu; Kangari; Kigumo; Kinyona; Muthithi; Gaturi; Mbiri; Mugoiri; Murarandia; Township; Wangu; Ichagaki; Kamahuha; Kambiti; Kimorori/Wempa; Makuyu; Nginda; Gitugi; Kamacharia; Kiru,Agriculture,Completed,3.0E7,3.0E7,3.0E7,100.0,FY 14-15,2014-06-01,2017-01-01,SIDA,SIDA: 30000000.00,null


In [0]:
-- Build a clean, queryable UC table from the raw CSV in the volume.
-- Cleaning applied:
--   1. multiLine => true : quoted fields containing newlines are parsed as one row (default parsing inflates 1129 rows to 1162)
--   2. snake_case column names
--   3. text normalization: embedded newlines/carriage returns collapsed, whitespace trimmed
--   4. completion_pct: values stored as fractions (<=1, e.g. 0.58) rescaled to 0-100
--   5. start_date/end_date: Excel artifacts (1905-xx-xx, 2108-xx-xx) set to NULL; valid range 1950-01-01..today
--   6. fiscal_year_start derived from 'FY 13-14' -> 2013 for sorting/filtering
CREATE OR REPLACE TABLE projects.muranga.projects
COMMENT 'Murang''a County government projects FY 2013/14 onward: budget, disbursement, expenditure, status and location by ward and sub-county. Cleaned from /Volumes/projects/muranga/projects/Murang''a_Projects.csv'
AS
SELECT
  trim(regexp_replace(`Project Name`, '\\s+', ' '))            AS project_name,
  trim(regexp_replace(`Description`, '\\s+', ' '))             AS description,
  trim(regexp_replace(`Sub-County(s)`, '\\s+', ' '))           AS sub_counties,
  trim(regexp_replace(`Ward(s)`, '\\s+', ' '))                 AS wards,
  trim(regexp_replace(`Sector(s)`, '\\s+', ' '))               AS sector,
  initcap(trim(`Status`))                                        AS status,
  cast(`Approved Cost` AS DECIMAL(18,2))                         AS approved_cost_kes,
  cast(`Disbursed`     AS DECIMAL(18,2))                         AS disbursed_kes,
  cast(`Expenditure`   AS DECIMAL(18,2))                        AS expenditure_kes,
  round(CASE WHEN `Completion %` <= 1
              THEN `Completion %` * 100
              ELSE `Completion %` END, 1)                        AS completion_pct,
  trim(`Fiscal Year`)                                            AS fiscal_year,
  cast('20' || regexp_extract(trim(`Fiscal Year`), 'FY (\\d{2})', 1) AS INT) AS fiscal_year_start,
  CASE WHEN `Start Date` BETWEEN date'1950-01-01' AND current_date()
       THEN `Start Date` END                                      AS start_date,
  CASE WHEN `End Date` BETWEEN date'1950-01-01' AND current_date()
       THEN `End Date` END                                        AS end_date,
  trim(`Development Partners`)                                    AS development_partners,
  trim(regexp_replace(`Partner Components`, '\\s+', ' '))        AS partner_components
FROM read_files('/Volumes/projects/muranga/projects', format => 'csv', multiLine => true)

num_affected_rows,num_inserted_rows


In [0]:
-- Document columns for discoverability in Catalog Explorer
ALTER TABLE projects.muranga.projects ALTER COLUMN project_name        COMMENT 'Name of the county project';
ALTER TABLE projects.muranga.projects ALTER COLUMN description         COMMENT 'What the project delivers';
ALTER TABLE projects.muranga.projects ALTER COLUMN sub_counties         COMMENT 'Semicolon-separated list of sub-counties covered';
ALTER TABLE projects.muranga.projects ALTER COLUMN wards               COMMENT 'Semicolon-separated list of wards covered';
ALTER TABLE projects.muranga.projects ALTER COLUMN sector              COMMENT 'County sector, e.g. Agriculture, Infrastructure, Education VTC';
ALTER TABLE projects.muranga.projects ALTER COLUMN status              COMMENT 'Project status: Completed, Planned, Ongoing, Stalled';
ALTER TABLE projects.muranga.projects ALTER COLUMN approved_cost_kes    COMMENT 'Approved budget in Kenyan shillings';
ALTER TABLE projects.muranga.projects ALTER COLUMN disbursed_kes        COMMENT 'Funds disbursed in Kenyan shillings';
ALTER TABLE projects.muranga.projects ALTER COLUMN expenditure_kes     COMMENT 'Funds spent in Kenyan shillings';
ALTER TABLE projects.muranga.projects ALTER COLUMN completion_pct       COMMENT 'Physical completion, 0-100. Values originally stored as fractions (e.g. 0.58) were rescaled to percent';
ALTER TABLE projects.muranga.projects ALTER COLUMN fiscal_year         COMMENT 'County fiscal year label, e.g. FY 13-14 (Jul-Jun)';
ALTER TABLE projects.muranga.projects ALTER COLUMN fiscal_year_start    COMMENT 'Calendar year the fiscal year started, derived from fiscal_year (FY 13-14 -> 2013)';
ALTER TABLE projects.muranga.projects ALTER COLUMN start_date          COMMENT 'Project start date. Excel artifacts (1905-xx-xx, 2108-xx-xx) nulled to NULL';
ALTER TABLE projects.muranga.projects ALTER COLUMN end_date            COMMENT 'Planned/projected end date. Excel artifacts (1905-xx-xx, 2108-xx-xx) set to NULL';
ALTER TABLE projects.muranga.projects ALTER COLUMN development_partners COMMENT 'External funding partner(s), e.g. IFAD, SIDA; NULL when county-funded';
ALTER TABLE projects.muranga.projects ALTER COLUMN partner_components  COMMENT 'Per-partner funding component detail, e.g. IFAD: 5000000.00'

In [0]:
-- Verify the clean table against expectations
SELECT count(*)                                   AS total_rows,
       count_if(status IS NULL)                   AS null_status,
       min(completion_pct)                        AS min_completion,
       max(completion_pct)                        AS max_completion,
       min(start_date)                            AS min_start_date,
       max(start_date)                            AS max_start_date,
       min(end_date)                             AS min_end_date,
       max(end_date)                             AS max_end_date,
       min(fiscal_year_start)                     AS min_fy,
       max(fiscal_year_start)                     AS max_fy,
       count_if(project_name <> trim(regexp_replace(project_name, '\\s+', ' '))) AS names_with_extra_ws
FROM projects.muranga.projects

total_rows,null_status,min_completion,max_completion,min_start_date,max_start_date,min_end_date,max_end_date,min_fy,max_fy,names_with_extra_ws
1129,0,0.0,100.0,2014-01-01,2026-08-07,2014-06-01,2026-10-13,2013,2025,0


In [0]:
SELECT * 
FROM projects.muranga.projects

project_name,description,sub_counties,wards,sector,status,approved_cost_kes,disbursed_kes,expenditure_kes,completion_pct,fiscal_year,fiscal_year_start,start_date,end_date,development_partners,partner_components
French beans and marketing project,Provide production inputs and improve market access for french beans,Gatanga; Kandara; Kangema; Kigumo; Kiharu; Maragua; Mathioya,Gatanga; Ithanga; Kakuzi/Mitubiri; Kariara; Kihumbu-ini; Mugumo-ini; Gaichanjiru; Ithiru; Kagundu-ini; Muruka; Ng'araria; Ruchu; Kanyenyaini; Muguru; Rwathia; Kahumbu; Kangari; Kigumo; Kinyona; Muthithi; Gaturi; Mbiri; Mugoiri; Murarandia; Township; Wangu; Ichagaki; Kamahuha; Kambiti; Kimorori/Wempa; Makuyu; Nginda; Gitugi; Kamacharia; Kiru,Agriculture,Planned,6000000.00,6000000.00,6000000.00,100.0,FY 13-14,2013,2014-01-01,2015-01-01,null,null
Kenyatta Agriculture Farmers Training Centre (Mariira farm ). Staff and farmers capacity building,Capacity building of staff and farmers on agricultural technologies,Gatanga; Kandara; Kangema; Kigumo; Kiharu; Maragua; Mathioya,Gatanga; Ithanga; Kakuzi/Mitubiri; Kariara; Kihumbu-ini; Mugumo-ini; Gaichanjiru; Ithiru; Kagundu-ini; Muruka; Ng'araria; Ruchu; Kanyenyaini; Muguru; Rwathia; Kahumbu; Kangari; Kigumo; Kinyona; Muthithi; Gaturi; Mbiri; Mugoiri; Murarandia; Township; Wangu; Ichagaki; Kamahuha; Kambiti; Kimorori/Wempa; Makuyu; Nginda; Gitugi; Kamacharia; Kiru,Agriculture,Ongoing,1120000.00,1120000.00,1120000.00,100.0,FY 13-14,2013,2014-07-01,2014-07-01,null,null
Gikindu (Thailand) Farmers Dry land Training Centre project,Project to develop support infrastructure for stakeholders training,Kiharu,Mbiri,Agriculture,Stalled,205000000.00,0.00,100000000.00,50.0,FY 13-14,2013,2014-06-01,2014-06-01,null,null
Semi-permanent French beans collection centres in Kigumo and Kandara,Establish infrastructure for aggregation of fresh farm produce in readiness for market offtake,Kandara; Kigumo,Ithiru; Kigumo,Agriculture,Completed,4000000.00,4000000.00,4000000.00,100.0,FY 14-15,2014,2014-01-01,2015-01-01,null,null
Gikindu (Thailand) Farmers Dry land Training Centre project,Project to develop support infrastructure for stakeholders training,Kiharu,Mbiri,Agriculture,Stalled,5000000.00,5000000.00,5000000.00,50.0,FY 14-15,2014,2014-06-01,2015-06-01,null,null
Ajibika Irrigation project,An initiative designed to convey water to small holder farms to boost food and nutrition security.,Kandara,Ithiru; Ruchu,Agriculture,Completed,100000000.00,100000000.00,100000000.00,100.0,FY 14-15,2014,2014-06-01,2015-01-01,null,null
Kanyenyaini Irrigation project,null,Kangema,Kanyenyaini,Agriculture,Stalled,87000000.00,87000000.00,87000000.00,58.0,FY 14-15,2014,2014-06-01,2015-01-01,null,null
Kimathi Githuri irrigation project-Food security programme (rice and Vegetables),An initiative designed to convey water to small holder farms to boost food and nutrition security.,Kiharu,Gaturi; Mbiri,Agriculture,Completed,10000000.00,10000000.00,10000000.00,100.0,FY 14-15,2014,null,2017-01-01,null,null
Food a nd nutrition security -Ria Komu- Thika Ridge irrigation Project,An initiative designed to convey water to small holder farms to boost food and nutrition security.,Gatanga,Kihumbu-ini,Agriculture,Completed,8000000.00,5000000.00,5000000.00,100.0,FY 14-15,2014,null,2019-01-01,IFAD,IFAD: 5000000.00
Agriculture Sector Development Support Programme Phase one,"Project contributed to transformation of Dairy ,French Beans and Banana Value chains to commercially oriented enterprises that ensure sustainable food and nutrition security.",Gatanga; Kandara; Kangema; Kigumo; Kiharu; Maragua; Mathioya,Gatanga; Ithanga; Kakuzi/Mitubiri; Kariara; Kihumbu-ini; Mugumo-ini; Gaichanjiru; Ithiru; Kagundu-ini; Muruka; Ng'araria; Ruchu; Kanyenyaini; Muguru; Rwathia; Kahumbu; Kangari; Kigumo; Kinyona; Muthithi; Gaturi; Mbiri; Mugoiri; Murarandia; Township; Wangu; Ichagaki; Kamahuha; Kambiti; Kimorori/Wempa; Makuyu; Nginda; Gitugi; Kamacharia; Kiru,Agriculture,Completed,30000000.00,30000

In [0]:
-- Session-scoped temp table with all columns except start_date and end_date.
-- Temp tables don't support CREATE OR REPLACE, so drop-if-exists first.
DROP TEMP TABLE IF EXISTS projects_no_dates;

CREATE TEMP TABLE projects_no_dates AS
SELECT
  project_name,
  description,
  sub_counties,
  wards,
  sector,
  status,
  approved_cost_kes,
  disbursed_kes,
  expenditure_kes,
  completion_pct,
  fiscal_year,
  fiscal_year_start,
  development_partners,
  partner_components
FROM projects.muranga.projects;

SELECT * FROM projects_no_dates

project_name,description,sub_counties,wards,sector,status,approved_cost_kes,disbursed_kes,expenditure_kes,completion_pct,fiscal_year,fiscal_year_start,development_partners,partner_components
French beans and marketing project,Provide production inputs and improve market access for french beans,Gatanga; Kandara; Kangema; Kigumo; Kiharu; Maragua; Mathioya,Gatanga; Ithanga; Kakuzi/Mitubiri; Kariara; Kihumbu-ini; Mugumo-ini; Gaichanjiru; Ithiru; Kagundu-ini; Muruka; Ng'araria; Ruchu; Kanyenyaini; Muguru; Rwathia; Kahumbu; Kangari; Kigumo; Kinyona; Muthithi; Gaturi; Mbiri; Mugoiri; Murarandia; Township; Wangu; Ichagaki; Kamahuha; Kambiti; Kimorori/Wempa; Makuyu; Nginda; Gitugi; Kamacharia; Kiru,Agriculture,Planned,6000000.00,6000000.00,6000000.00,100.0,FY 13-14,2013,null,null
Kenyatta Agriculture Farmers Training Centre (Mariira farm ). Staff and farmers capacity building,Capacity building of staff and farmers on agricultural technologies,Gatanga; Kandara; Kangema; Kigumo; Kiharu; Maragua; Mathioya,Gatanga; Ithanga; Kakuzi/Mitubiri; Kariara; Kihumbu-ini; Mugumo-ini; Gaichanjiru; Ithiru; Kagundu-ini; Muruka; Ng'araria; Ruchu; Kanyenyaini; Muguru; Rwathia; Kahumbu; Kangari; Kigumo; Kinyona; Muthithi; Gaturi; Mbiri; Mugoiri; Murarandia; Township; Wangu; Ichagaki; Kamahuha; Kambiti; Kimorori/Wempa; Makuyu; Nginda; Gitugi; Kamacharia; Kiru,Agriculture,Ongoing,1120000.00,1120000.00,1120000.00,100.0,FY 13-14,2013,null,null
Gikindu (Thailand) Farmers Dry land Training Centre project,Project to develop support infrastructure for stakeholders training,Kiharu,Mbiri,Agriculture,Stalled,205000000.00,0.00,100000000.00,50.0,FY 13-14,2013,null,null
Semi-permanent French beans collection centres in Kigumo and Kandara,Establish infrastructure for aggregation of fresh farm produce in readiness for market offtake,Kandara; Kigumo,Ithiru; Kigumo,Agriculture,Completed,4000000.00,4000000.00,4000000.00,100.0,FY 14-15,2014,null,null
Gikindu (Thailand) Farmers Dry land Training Centre project,Project to develop support infrastructure for stakeholders training,Kiharu,Mbiri,Agriculture,Stalled,5000000.00,5000000.00,5000000.00,50.0,FY 14-15,2014,null,null
Ajibika Irrigation project,An initiative designed to convey water to small holder farms to boost food and nutrition security.,Kandara,Ithiru; Ruchu,Agriculture,Completed,100000000.00,100000000.00,100000000.00,100.0,FY 14-15,2014,null,null
Kanyenyaini Irrigation project,null,Kangema,Kanyenyaini,Agriculture,Stalled,87000000.00,87000000.00,87000000.00,58.0,FY 14-15,2014,null,null
Kimathi Githuri irrigation project-Food security programme (rice and Vegetables),An initiative designed to convey water to small holder farms to boost food and nutrition security.,Kiharu,Gaturi; Mbiri,Agriculture,Completed,10000000.00,10000000.00,10000000.00,100.0,FY 14-15,2014,null,null
Food a nd nutrition security -Ria Komu- Thika Ridge irrigation Project,An initiative designed to convey water to small holder farms to boost food and nutrition security.,Gatanga,Kihumbu-ini,Agriculture,Completed,8000000.00,5000000.00,5000000.00,100.0,FY 14-15,2014,IFAD,IFAD: 5000000.00
Agriculture Sector Development Support Programme Phase one,"Project contributed to transformation of Dairy ,French Beans and Banana Value chains to commercially oriented enterprises that ensure sustainable food and nutrition security.",Gatanga; Kandara; Kangema; Kigumo; Kiharu; Maragua; Mathioya,Gatanga; Ithanga; Kakuzi/Mitubiri; Kariara; Kihumbu-ini; Mugumo-ini; Gaichanjiru; Ithiru; Kagundu-ini; Muruka; Ng'araria; Ruchu; Kanyenyaini; Muguru; Rwathia; Kahumbu; Kangari; Kigumo; Kinyona; Muthithi; Gaturi; Mbiri; Mugoiri; Murarandia; Township; Wangu; Ichagaki; Kamahuha; Kambiti; Kimorori/Wempa; Makuyu; Nginda; Gitugi; Kamacharia; Kiru,Agriculture,Completed,30000000.00,30000000.00,30000000.00,100.0,FY 14-15,2014,SIDA,SIDA: 30000000.00


In [0]:
-- Performance by sector: budget vs disbursement vs spend
SELECT sector,
       count(*)                                              AS projects,
       count_if(status = 'Stalled')                          AS stalled,
       round(sum(approved_cost_kes)  / 1e9, 2)              AS approved_bn_kes,
       round(sum(disbursed_kes)      / 1e9, 2)               AS disbursed_bn_kes,
       round(sum(expenditure_kes)    / 1e9, 2)               AS spent_bn_kes,
       round(100 * sum(expenditure_kes) / nullif(sum(disbursed_kes), 0), 1) AS spend_vs_disbursed_pct,
       round(avg(completion_pct), 1)                         AS avg_completion_pct,
       round(avg(approved_cost_kes) / 1e6, 2)                AS avg_cost_per_project_m_kes
FROM projects.muranga.projects
GROUP BY sector
ORDER BY approved_bn_kes DESC

sector,projects,stalled,approved_bn_kes,disbursed_bn_kes,spent_bn_kes,spend_vs_disbursed_pct,avg_completion_pct,avg_cost_per_project_m_kes
Agriculture,104,3,8.45,10.05,7.33,73.0,75.9,81.23
Infrastructure,441,0,1.74,1.74,1.32,75.9,100.0,3.95
Trade,89,0,1.16,0.36,0.0,0.0,100.0,13.03
Irrigation,44,0,1.09,1.09,0.0,0.0,100.0,24.77
Lands,37,0,0.77,0.42,0.45,107.8,39.5,20.81
Education,214,0,0.55,0.55,0.5,90.9,100.0,2.59
Health,123,1,0.48,0.33,0.07,20.8,25.3,3.93
Education VTC,61,3,0.07,0.07,0.07,100.0,98.3,1.08
Environment,2,0,0.06,0.06,0.0,0.0,100.0,30.0
Municipality,5,0,0.06,0.06,0.06,98.6,80.0,11.6


In [0]:
-- Fiscal-year trend: how much budgeted vs disbursed vs spent each year
SELECT fiscal_year_start                            AS fy_start,
       count(*)                                     AS projects,
       round(sum(approved_cost_kes) / 1e9, 2)       AS approved_bn_kes,
       round(sum(disbursed_kes)     / 1e9, 2)       AS disbursed_bn_kes,
       round(sum(expenditure_kes)   / 1e9, 2)       AS spent_bn_kes,
       round(100 * sum(expenditure_kes) / nullif(sum(disbursed_kes), 0), 1) AS spend_vs_disbursed_pct,
       round(avg(completion_pct), 1)                AS avg_completion_pct,
       round(avg(approved_cost_kes) / 1e6, 2)               AS avg_cost_per_project_m_kes
FROM projects.muranga.projects
GROUP BY fiscal_year_start
ORDER BY fiscal_year_start

fy_start,projects,approved_bn_kes,disbursed_bn_kes,spent_bn_kes,spend_vs_disbursed_pct,avg_completion_pct,avg_cost_per_project_m_kes
2013,11,0.81,0.61,0.11,17.7,95.5,73.73
2014,13,0.77,0.8,0.8,100.0,92.9,59.15
2015,35,0.61,0.56,0.33,59.1,71.7,17.49
2016,27,1.33,0.5,0.49,97.9,96.3,49.42
2017,23,0.88,1.01,0.83,81.8,91.3,38.22
2018,15,1.05,0.97,0.76,78.4,93.3,70.19
2019,54,1.21,0.97,0.94,97.7,93.7,22.5
2020,43,0.93,2.83,0.76,26.9,92.3,21.68
2021,127,0.75,0.75,0.67,89.8,96.0,5.89
2022,158,1.21,2.1,1.8,85.8,94.9,7.67


In [0]:
-- Budget overruns: expenditure more than 5% above approved cost
SELECT project_name,
       sector,
       fiscal_year,
       status,
       approved_cost_kes,
       expenditure_kes,
       round(expenditure_kes - approved_cost_kes, 0) AS overrun_kes,
       round(100 * expenditure_kes / nullif(approved_cost_kes, 0), 1) AS pct_of_budget
FROM projects.muranga.projects
WHERE approved_cost_kes > 0
  AND expenditure_kes > approved_cost_kes * 1.05
ORDER BY overrun_kes DESC
LIMIT 15

project_name,sector,fiscal_year,status,approved_cost_kes,expenditure_kes,overrun_kes,pct_of_budget
Agriculture Sector Development Support Programme Phase Two,Agriculture,FY 15-16,Completed,21000.00,21000000.00,20979000,100000.0
Ititu Ikundu Irrigation project,Agriculture,FY 21-22,Completed,23000000.00,25000000.00,2000000,108.7


In [0]:
-- Stalled and low-completion projects: money spent vs work delivered
SELECT project_name,
       sector,
       fiscal_year,
       status,
       approved_cost_kes,
       expenditure_kes,
       completion_pct
FROM projects.muranga.projects
WHERE status IN ('Stalled', 'Ongoing')
   OR (status NOT IN ('Completed') AND expenditure_kes > 0 AND completion_pct < 50)
ORDER BY expenditure_kes / nullif(approved_cost_kes, 0) DESC, expenditure_kes DESC

project_name,sector,fiscal_year,status,approved_cost_kes,expenditure_kes,completion_pct
Certified maize seeds and fertilizer subsidy programme,Agriculture,FY 23-24,Planned,220000000.00,220000000.00,0.0
Certified maize seeds and fertilizer subsidy programme,Agriculture,FY 22-23,Planned,220000000.00,220000000.00,0.0
National Agricultyral Value Chains Development Project (NAVCDP),Agriculture,FY 23-24,Planned,199798000.00,199798000.00,0.0
Food and nutrition security,Agriculture,FY 19-20,Ongoing,188000000.00,188000000.00,100.0
Milk and Mango e-voucher subsidy progrmme,Agriculture,FY 22-23,Planned,156000000.00,156000000.00,0.0
Food and nutrition security project,Agriculture,FY 20-21,Ongoing,100000000.00,100000000.00,100.0
Kanyenyaini Irrigation project,Agriculture,FY 14-15,Stalled,87000000.00,87000000.00,58.0
National Agricultyral Value Chains Development Project (NAVCDP),Agriculture,FY 22-23,Planned,67192729.00,67192729.00,0.0
Food and nutrition security project,Agriculture,FY 21-22,Ongoing,22200000.00,22200000.00,0.0
Food and nutrition security,Agriculture,FY 16-17,Ongoing,17000000.00,17000000.00,100.0


In [0]:
-- Exploded geography views for ward/sub-county visualizations.
-- `wards` and `sub_counties` hold semicolon-separated lists (one project can cover many geographies),
-- so each project is exploded into one row per geography and its money is split evenly
-- across the listed geographies (share_*_kes) to avoid double-counting spend in geo charts.
-- Projects with no location listed are absent from these views (they still appear in projects.muranga.projects).

CREATE OR REPLACE VIEW projects.muranga.projects_by_ward AS
SELECT project_name,
       sector,
       status,
       fiscal_year,
       fiscal_year_start,
       completion_pct,
       approved_cost_kes,
       disbursed_kes,
       expenditure_kes,
       trim(w.ward) AS ward,
       cast(approved_cost_kes / nullif(p.n_geos, 0) AS DOUBLE) AS share_approved_kes,
       cast(disbursed_kes      / nullif(p.n_geos, 0) AS DOUBLE) AS share_disbursed_kes,
       cast(expenditure_kes    / nullif(p.n_geos, 0) AS DOUBLE) AS share_expenditure_kes
FROM (SELECT *, size(filter(split(wards, ';'), x -> trim(x) <> '')) AS n_geos
      FROM projects.muranga.projects) p
LATERAL VIEW explode(split(p.wards, ';')) w AS ward
WHERE trim(w.ward) <> '';

CREATE OR REPLACE VIEW projects.muranga.projects_by_sub_county AS
SELECT project_name,
       sector,
       status,
       fiscal_year,
       fiscal_year_start,
       completion_pct,
       approved_cost_kes,
       disbursed_kes,
       expenditure_kes,
       trim(sc.sub_county) AS sub_county,
       cast(approved_cost_kes / nullif(p.n_geos, 0) AS DOUBLE) AS share_approved_kes,
       cast(disbursed_kes      / nullif(p.n_geos, 0) AS DOUBLE) AS share_disbursed_kes,
       cast(expenditure_kes    / nullif(p.n_geos, 0) AS DOUBLE) AS share_expenditure_kes
FROM (SELECT *, size(filter(split(sub_counties, ';'), x -> trim(x) <> '')) AS n_geos
      FROM projects.muranga.projects) p
LATERAL VIEW explode(split(p.sub_counties, ';')) sc AS sub_county
WHERE trim(sc.sub_county) <> ''